# 🚨 PROJECT OVERFIT — Student Workbook

**EC34: Data Science for Economists**

Open the game app alongside this notebook:
```
uv run streamlit run games/ml/app.py
```

For each module:
1. Read the mission brief in the app
2. Fill in the `___` blanks in the cell below
3. Run the cell and inspect the output
4. Enter your answer in the app to unlock the next module

> **Rules:** Every blank requires a decision. Think about *why* before filling in *what*.

---

In [ ]:
# Run if needed:
# !pip install scikit-learn

In [ ]:
# Run this cell first — imports for the whole notebook
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_regression, make_blobs
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

---
## Module 1 — The Leaky Pipeline

The intern selected features by correlating them with `y` — but did they do it before or after the train/test split?

Below are two pipelines. Each has one blank at the critical decision point.
Fill in what data each pipeline uses to compute the correlations, then run and compare.

**The question asks for Pipeline A's test R².**

In [ ]:
np.random.seed(99)
X = np.random.randn(100, 500)   # 500 features — ALL pure noise
y = np.random.randn(100)        # target — also pure noise

# ── PIPELINE A  (leaky) ──────────────────────────────────────────────────────
# Feature selection using ALL 100 observations — before the split.
# Fill in the blank: which dataset should go here to reproduce the leaky pipeline?
corrs_a = np.abs(np.corrcoef(___.T, y)[-1, :-1])   # <-- fill in: X  or  X_tr_a ?
top_idx_a = np.argsort(corrs_a)[-25:]
X_sel_a = X[:, top_idx_a]

X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(X_sel_a, y, test_size=0.3, random_state=7)
r2_a = r2_score(y_te_a, LinearRegression().fit(X_tr_a, y_tr_a).predict(X_te_a))

# ── PIPELINE B  (correct) ────────────────────────────────────────────────────
# Split first, then select features using training data only.
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X, y, test_size=0.3, random_state=7)

# Fill in the two blanks: which X and which y to use for correlations?
corrs_b = np.abs(np.corrcoef(___.T, ___)[-1, :-1])  # <-- fill in
top_idx_b = np.argsort(corrs_b)[-25:]
r2_b = r2_score(
    y_te_b,
    LinearRegression().fit(X_tr_b[:, top_idx_b], y_tr_b).predict(X_te_b[:, top_idx_b])
)

print(f"Pipeline A (leaky)   test R²: {r2_a:.2f}")
print(f"Pipeline B (correct) test R²: {r2_b:.2f}")
print()
print("Both datasets contain ZERO true signal.")
print("Any positive R² from Pipeline A is entirely fabricated by the leak.")

**Submit to the app:** Pipeline A's test R² (2 decimal places)

---
## Module 2 — The High-Dimensional Trap

200 features, 100 observations. OLS memorises the training data completely.
Your job: fit LassoCV *correctly* (scale on train only) and count how many features survive.

The blanks test:
- What do you fit the scaler on?
- What condition defines a "non-zero" Lasso coefficient?

In [ ]:
np.random.seed(0)
X, y = make_regression(noise=4, random_state=0, n_samples=100, n_features=200)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=123)

# ── Scaling ───────────────────────────────────────────────────────────────────
# Fit the scaler on training data only — never on the full dataset.
# Fill in the blank: what do you pass to .fit()?
scaler = StandardScaler().fit(___)          # <-- fill in
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

# ── Lasso ─────────────────────────────────────────────────────────────────────
lasso = LassoCV(alphas=np.linspace(0.1, 1, 30), random_state=0, max_iter=10000)
lasso.fit(X_tr_s, y_tr)

# ── Count surviving features ───────────────────────────────────────────────────
# Lasso sets irrelevant coefficients to *exactly* 0.
# Fill in the comparison operator and threshold to count non-zero coefficients.
n_nonzero = (np.abs(lasso.coef_) ___ ___).sum()    # <-- fill in: operator and threshold

print(f"Optimal alpha  : {lasso.alpha_:.3f}")
print(f"Non-zero coefs : {n_nonzero} out of 200")
print(f"Train R²       : {lasso.score(X_tr_s, y_tr):.3f}")
print(f"Test  R²       : {lasso.score(X_te_s, y_te):.3f}")

**Submit to the app:** The number of non-zero Lasso coefficients

---
## Module 3 — Tree Depth Disaster

The intern set `max_depth=20`. You need to find the right depth using 5-fold cross-validation.

The blanks test:
- What goes inside `cross_val_score` — estimator, folds, metric?
- How do you extract the mean score from the output?
- How do you pick the best depth from a dictionary?

In [ ]:
np.random.seed(1)
X = np.sort(5 * np.random.rand(80, 1), axis=0)
y = np.sin(X).ravel()
y[::5] += 3 * (0.5 - np.random.rand(16))

cv_scores = {}
for depth in range(1, 11):
    scores = cross_val_score(
        DecisionTreeRegressor(max_depth=___),   # <-- fill in: which depth to try?
        X, y,
        cv=___,                                  # <-- fill in: how many folds?
        scoring=___                              # <-- fill in: what metric? (hint: we want R²)
    )
    cv_scores[depth] = scores.___()             # <-- fill in: mean or something else?
    print(f"max_depth={depth:2d}  mean CV R² = {cv_scores[depth]:.4f}")

# Find the depth with the highest CV score
best_depth = ___(cv_scores, key=cv_scores.get)  # <-- fill in: max or min?
print(f"\nBest max_depth: {best_depth}")

In [ ]:
# Optional: visualise the validation curve
plt.figure(figsize=(8, 4))
plt.plot(list(cv_scores.keys()), list(cv_scores.values()), 'o-')
plt.axvline(best_depth, color='red', linestyle='--', label=f'Best depth = {best_depth}')
plt.xlabel('max_depth'); plt.ylabel('Mean 5-fold CV R²')
plt.title('Validation curve: decision tree depth')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Submit to the app:** The best `max_depth` value

---
## Module 4 — The Cluster Conundrum

195 countries, one cluster. You need to find the right k from the elbow plot.

The blanks test:
- Can you fill in the KMeans loop body correctly?
- Can you read your own elbow plot and pick k?
- Do you know which attribute holds inertia?

In [ ]:
np.random.seed(42)
X_raw, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.9, random_state=42)

# Standardise — always required before K-Means
X = StandardScaler().fit_transform(X_raw)

# ── Elbow plot ────────────────────────────────────────────────────────────────
inertias = {}
for k in range(1, 9):
    km = KMeans(n_clusters=___, random_state=42, n_init="auto")  # <-- fill in
    km.fit(X)
    inertias[k] = km.___                                          # <-- fill in: which attribute?

plt.figure(figsize=(7, 4))
plt.plot(list(inertias.keys()), list(inertias.values()), 'o-')
plt.xlabel('k (number of clusters)'); plt.ylabel('Inertia')
plt.title('Elbow Plot — where does the curve bend?')
plt.xticks(range(1, 9)); plt.grid(alpha=0.3)
plt.show()

# ── Fit with the optimal k ────────────────────────────────────────────────────
# Look at the elbow plot above. Where does the curve stop dropping sharply?
best_k = ___                                                    # <-- fill in: your answer from the plot

km_best = KMeans(n_clusters=best_k, random_state=42, n_init="auto").fit(X)
print(f"Inertia at k={best_k}: {km_best.inertia_:.0f}")
print(f"Cluster sizes: {np.bincount(km_best.labels_)}")

**Submit to the app:** The inertia at your chosen `best_k`, rounded to the nearest whole number

---
## Module 5 — Bagging from Scratch

A single fully-grown tree memorises everything. Bagging fixes it by averaging many trees
each trained on a **bootstrap sample** — a sample with replacement of the same size.

The blanks test:
- What are the arguments to `np.random.randint` for a bootstrap sample?
- Should the individual trees be shallow or fully grown? Why?
- How do you combine the 50 predictions into one?

In [ ]:
np.random.seed(2026)
X = np.sort(5 * np.random.rand(80, 1), axis=0)
y = np.sin(X).ravel() + np.random.normal(0, 0.2, 80)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=7)

# ── Baseline: single fully-grown tree ────────────────────────────────────────
single = DecisionTreeRegressor(max_depth=None).fit(X_tr, y_tr)
print(f"Single tree  train R²: {r2_score(y_tr, single.predict(X_tr)):.3f}")
print(f"Single tree  test  R²: {r2_score(y_te, single.predict(X_te)):.3f}")

# ── Bagging: 50 bootstrap trees ──────────────────────────────────────────────
np.random.seed(0)                              # do not change this seed!
bag_preds = np.zeros((len(X_te), 50))

for i in range(50):
    # Bootstrap sample: draw len(X_tr) indices WITH replacement from [0, len(X_tr))
    # np.random.randint(low, high, size) — what are low, high, and size here?
    idx = np.random.randint(___, ___, ___)     # <-- fill in: low, high, size

    # Fit a tree on the bootstrap sample.
    # Should it be shallow (low variance) or fully grown (no bias)? Why?
    t = DecisionTreeRegressor(max_depth=___).fit(X_tr[idx], y_tr[idx])  # <-- fill in

    bag_preds[:, i] = t.predict(X_te)

# Combine the 50 predictions — what operation reduces variance across trees?
bagged_pred = bag_preds.___(axis=1)            # <-- fill in: what aggregation?

r2_bag = r2_score(y_te, bagged_pred)
print(f"\nBagged ensemble (50 trees) test R²: {r2_bag:.3f}")

In [ ]:
# Optional: visualise single tree vs bagged ensemble
X_plot = np.linspace(X_tr.min(), X_tr.max(), 300).reshape(-1, 1)

np.random.seed(0)
bag_plot = np.zeros((300, 50))
for i in range(50):
    idx = np.random.randint(0, len(X_tr), len(X_tr))
    t = DecisionTreeRegressor(max_depth=None).fit(X_tr[idx], y_tr[idx])
    bag_plot[:, i] = t.predict(X_plot).ravel()

plt.figure(figsize=(10, 4))
plt.scatter(X_tr, y_tr, s=20, color='gray', alpha=0.5, label='Train')
plt.scatter(X_te, y_te, s=30, color='black', alpha=0.8, label='Test')
plt.plot(X_plot, single.predict(X_plot), color='red', lw=1.5, alpha=0.7, label='Single tree')
plt.plot(X_plot, bag_plot.mean(axis=1), color='steelblue', lw=2.5, label='Bagged (50 trees)')
plt.plot(X_plot, np.sin(X_plot), color='green', lw=1, linestyle=':', alpha=0.6, label='True signal')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Single tree vs bagged ensemble')
plt.show()

**Submit to the app:** The test R² of the bagged ensemble (3 decimal places)